# BERN02 Generalised Linear Models
# Haoyang Qian
# 2026/9/8

In [1]:
#!mamba install pandas

In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [3]:
df=pd.read_csv("bird_count.csv")

In [4]:
df["x"]=df["yr"]-1999
y=df["count"].values
x=df["x"].values

In [5]:
print("===dataframe===")
print(df)

===dataframe===
    count    yr  observerAge   lineCov   x
0       2  2011           63  1.000000  12
1       5  2010           64  0.987500  11
2      12  2002           30  0.991667   3
3      12  2006           60  0.987500   7
4       5  2008           62  0.987500   9
5      12  2012           64  1.000000  13
6       5  2000           28  0.991667   1
7       8  2005           59  0.987500   6
8       9  2007           61  0.987500   8
9      14  2004           58  1.000000   5
10     10  1999           27  0.991667   0
11      9  2001           29  0.991667   2
12      5  2009           63  0.987500  10


In [8]:
def poisson(params,x,y):
    beta_0,beta_1=params
    lambda_=np.exp(beta_0+beta_1*x)
    log_likelihood=sum(lambda_-y*(beta_0+beta_1*x))
            #-np.log(math.factorial(y))，this part is ignored as it doesn't have impact on the minimization result.
    return log_likelihood


#parameter optimization

initial_guess = [0.0, 0.0]
result = minimize(poisson, initial_guess, args=(x, y), method='BFGS')
beta_0_hat,beta_1_hat=result.x
print(f"optimal beta_0_hat and beta_1_hat are {beta_0_hat:.2f}, {beta_1_hat:.2f},respectively")


#generate three samples

df["lambda_hat"]=np.exp(beta_0_hat+beta_1_hat*x)
np.random.seed(5)
df["prediction_1"]=np.random.poisson(lam=df["lambda_hat"])
np.random.seed(7)
df["prediction_2"]=np.random.poisson(lam=df["lambda_hat"])
np.random.seed(8)
df["prediction_3"]=np.random.poisson(lam=df["lambda_hat"])

df.to_csv("bird_count_predictions.csv", index=False)

optimal beta_0_hat and beta_1_hat are 2.33, -0.03,respectively


### Model Discussion

The estimated parameters are $\hat{\beta}_0 = 2.32$ and $\hat{\beta}_1 = -0.03$. The negative coefficient $\hat{\beta}_1$ indicates a slight downward trend in bird populations over the observed period. 

Based on the estimated expected values ($\hat{\lambda}$), three hypothetical datasets (`prediction_1`, `prediction_2`, and `prediction_3`) were generated via random Poisson sampling. The simulated values exhibit natural sampling variation around $\hat{\lambda}$, reflecting expected ecological randomness.

In [7]:
print("===dataframe with predicted data===")
print(df)

===dataframe with predicted data===
    count    yr  observerAge   lineCov   x  lambda_hat  prediction_1  \
0       2  2011           63  1.000000  12    6.931067             9   
1       5  2010           64  0.987500  11    7.159620             6   
2      12  2002           30  0.991667   3    9.281307             8   
3      12  2006           60  0.987500   7    8.151726             5   
4       5  2008           62  0.987500   9    7.639585             4   
5      12  2012           64  1.000000  13    6.709810             7   
6       5  2000           28  0.991667   1    9.903506             9   
7       8  2005           59  0.987500   6    8.420531            12   
8       9  2007           61  0.987500   8    7.891502             6   
9      14  2004           58  1.000000   5    8.698200            10   
10     10  1999           27  0.991667   0   10.230076             9   
11      9  2001           29  0.991667   2    9.587360             9   
12      5  2009           63